# Setup

In [ ]:
!pip uninstall numpy -y
!pip install surprise
!pip install "numpy<2" --force-reinstall

import os
os.kill(os.getpid(), 9)

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 96.4 MB/s eta 0:00:00
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl size=2505216 sha256=3a7ff472c84d74c24bffcab84f4be9c98c8657b88cbe733203731106fe8ef031
  Stored in directory: /root/.cache/pip/wheels/2a/8f/6e/7e2899163e2d85d8266daab4aa1cdabec7a6c56f83c015b5af
Successfully built scikit-surprise
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 re

In [1]:
import random
import tqdm

import numpy as np
import pandas as pd

from surprise import Dataset
from datetime import datetime

# Data Prep

In [2]:
data = Dataset.load_builtin('ml-1m')
ratings_df = pd.DataFrame(data.raw_ratings, columns=["userId", "movieId", "rating", "tstamp"])
rating_matrix = ratings_df.pivot(index='userId', columns='movieId', values='rating')

user_ids = ratings_df['userId'].unique().tolist()
movie_ids = ratings_df['movieId'].unique().tolist()

Dataset ml-1m could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-1m.zip...
Done! Dataset ml-1m has been saved to /root/.surprise_data/ml-1m


In [3]:
# Generate a random list of k ratings for each user
def sample_user_ratings(k = 1):
  user_sample_ratings = {}

  for user_id in tqdm.tqdm(ratings_df['userId'].unique()):
    user_ratings = ratings_df[ratings_df['userId'] == user_id]

    num_samples = min(k, len(user_ratings)) # if user has less than k ratings, this ensures all are sampled

    sample_ratings = user_ratings.sample(n=num_samples)

    user_sample_ratings[user_id] = sample_ratings.to_dict('records')

  return user_sample_ratings # a dictionary mapping user ids to lists of ratings

sample_sizes = [1, 5, 10]
user_samples = {k: sample_user_ratings(k) for k in sample_sizes}

100%|██████████| 6040/6040 [09:44<00:00, 10.33it/s]


In [4]:
def timestamp_to_year_month(timestamp):
  timestamp = int(timestamp)
  dt_object = datetime.fromtimestamp(timestamp)
  return dt_object.year, dt_object.month

# Candidate Selection

In [5]:
# Epsilon is a tolerance variable. It describe's the user's uncertainty of rating value
# High epsilon values mean a less restrictive filter on candidate users
# Low epsilon values mean a more restrictive filter on candidate users - 0 indicates absolute certainty
epsilons = [0, 0.5, 1]

# Finds all users that have rated movie_id within epsilon rating at time year, month
def find_candidates(movie_id, rating, year, month, epsilon = 0):
  # Filter by movie ID and rating
  filtered_df = ratings_df[
      (ratings_df['movieId'] == movie_id) &
      (ratings_df['rating'] >= rating - epsilon) &
      (ratings_df['rating'] <= rating + epsilon)
  ].copy()

  # Convert timestamps to year and month
  filtered_df['year'], filtered_df['month'] = zip(*filtered_df['tstamp'].apply(lambda x: timestamp_to_year_month(x)))

  # Filter by year and month
  filtered_df = filtered_df[(filtered_df['year'] == year) & (filtered_df['month'] == month)]

  # Get unique user IDs
  candidate_users = filtered_df['userId'].unique().tolist()

  return candidate_users # list of candidate user ids

# Identifies a most likely candidate user for a single user, given a sample of their ratings
def find_best_candidate(user_id, sample_ratings, epsilon = 0):
  candidate_ids = set(user_ids)

  for sample_rating in sample_ratings:
    movie_id = sample_rating['movieId']
    rating = sample_rating['rating']
    timestamp = int(sample_rating['tstamp'])
    year, month = timestamp_to_year_month(timestamp)

    candidate_ids = set(find_candidates(movie_id, rating, year, month, epsilon)) & candidate_ids

    if len(candidate_ids) == 0: # empty set, no candidates users, select random user
      best_candidate = random.choice(user_ids)
      break
    elif len(candidate_ids) == 1: # exactly one candidates user
      best_candidate = candidate_ids.pop()
      break

  # multiple candidates users, choose random from list
  if len(candidate_ids) > 1:
    best_candidate = random.choice(list(candidate_ids))

  return best_candidate

# Identifies the most likely candidate for each user and sample combination
def find_best_candidates(user_samples, epsilon = 0):
  best_candidates = {} # maps target user id to best guess

  for (user_id, sample_ratings) in tqdm.tqdm(user_samples.items()):
    best_candidate = find_best_candidate(user_id, sample_ratings)
    best_candidates[user_id] = best_candidate

  return best_candidates

# Execution & Evaluation

In [6]:
# Returns the ratio of correct candidates to total candidates
def evaluate_candidates(best_candidates):
  total_candidates = len(best_candidates)
  total_correct = 0
  for (user_id, best_candidate) in best_candidates.items():
    if user_id == best_candidate:
      total_correct += 1

  return total_correct / total_candidates

In [7]:
results = {}
for k in sample_sizes:
  user_samples_k = user_samples[k]
  epsilon_results = {}
  for epsilon in epsilons:
    best_candidates = find_best_candidates(user_samples_k, epsilon)
    score = evaluate_candidates(best_candidates)
    epsilon_results.update({epsilon: f"{score * 100:.2f}%"})
  results.update({k : epsilon_results})

100%|██████████| 6040/6040 [30:51<00:00,  3.26it/s]


In [8]:
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df.index.name = 'Sample Size (k)'
results_df.columns.name = 'Epsilon'
display("Percentage Correct for Reidentification")
display(results_df)

'Percentage Correct for Reidentification'

Epsilon,0.0,0.5,1.0
Sample Size (k),,,
1,12.75%,12.27%,12.67%
5,99.19%,99.07%,99.04%
10,100.00%,100.00%,100.00%
